In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform, randint # For RandomizedSearchCV distributions
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

# --- 1. Load the Actual Titanic Dataset ---
# IMPORTANT: Make sure 'titanic.csv' is in the same directory as your script,
# or provide the full path to the file.
try:
    df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
    print("Actual Titanic dataset loaded from 'titanic.csv'.")
    print(f"Dataset shape: {df.shape}")
    print("First 5 rows of the dataset:")
    print(df.head())
except FileNotFoundError:
    print("Error: 'titanic.csv' not found.")
    print("Please ensure the 'titanic.csv' file is in the correct directory.")
    print("You can download it from Kaggle's Titanic competition: https://www.kaggle.com/c/titanic/data")
    exit() # Exit if the file is not found, as we cannot proceed without actual data.

# --- 2. Preprocessing Steps ---
print("\nStarting data preprocessing...")

# Drop unnecessary columns
# 'PassengerId' is just an identifier. 'Name', 'Ticket', 'Cabin' often have
# too many unique values or missing values to be directly useful without
# more complex feature engineering, which is beyond the scope here.
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
print("Dropped 'PassengerId', 'Name', 'Ticket', 'Cabin' columns.")

# Define features (X) and target (y)
# 'Survived' is our target variable (what we want to predict).
X = df.drop('Survived', axis=1)
y = df['Survived']
print("Defined features (X) and target (y).")

# Identify numerical and categorical features for separate processing
numerical_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include='object').columns.tolist()

print(f"Numerical features identified: {numerical_features}")
print(f"Categorical features identified: {categorical_features}")

# Create preprocessing pipelines for numerical and categorical features
# Numerical: Impute missing values (e.g., in 'Age', 'Fare') with the median,
# then scale them to have zero mean and unit variance. Scaling is important
# for algorithms sensitive to feature magnitudes like Logistic Regression.
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Fills missing numbers with the column's median
    ('scaler', StandardScaler())                    # Scales numerical values
])
print("Created numerical preprocessing pipeline (median imputation, standard scaling).")

# Categorical: Impute missing values (e.g., in 'Embarked') with the most frequent value,
# then convert categorical text data into numerical format using one-hot encoding.
# 'handle_unknown='ignore'' ensures it doesn't break if an unseen category appears in test data.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fills missing categories with the most common one
    ('onehot', OneHotEncoder(handle_unknown='ignore'))     # Converts categories to numerical (binary) columns
])
print("Created categorical preprocessing pipeline (most frequent imputation, one-hot encoding).")

# Create a preprocessor using ColumnTransformer
# This allows applying different transformers to different columns.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),     # Apply numerical_transformer to numerical columns
        ('cat', categorical_transformer, categorical_features) # Apply categorical_transformer to categorical columns
    ])
print("Created ColumnTransformer for combined preprocessing.")

# Split data into training and testing sets
# We hold out 20% of the data for final testing to evaluate the model's performance
# on data it has never seen. random_state ensures reproducibility of the split.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data split into training (X_train shape: {X_train.shape}) and testing (X_test shape: {X_test.shape}).")
print("Preprocessing setup complete.")

# --- 3. Model Definition and Hyperparameter Optimization ---

# Define a function to evaluate and report model performance
# This function calculates and prints several common classification metrics.
def evaluate_model(model, X_test, y_test, model_name="Model"):
    """
    Evaluates a trained machine learning model on the test set and prints
    key performance metrics.

    Args:
        model: The trained scikit-learn model or pipeline.
        X_test: Features of the test set.
        y_test: True labels of the test set.
        model_name: A string name for the model (e.g., "Logistic Regression").

    Returns:
        A dictionary containing the calculated metrics.
    """
    y_pred = model.predict(X_test)
    # Get probabilities for ROC-AUC. `predict_proba` returns probabilities for each class.
    # We take the probability of the positive class (survival, index 1).
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    # Calculate standard classification metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else 'N/A'

    print(f"\n--- {model_name} Performance on Test Set ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1, "roc_auc": roc_auc}


print("\n--- Starting Hyperparameter Optimization and Validation ---")

# --- Logistic Regression Optimization ---
print("\n### Logistic Regression Hyperparameter Tuning ###")

# 1. Create a pipeline for Logistic Regression
# This pipeline integrates the preprocessing steps with the Logistic Regression classifier.
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('classifier', LogisticRegression(random_state=42))])
print("Logistic Regression pipeline created (Preprocessing + Classifier).")

# 2. Define parameter grid for Grid Search (Logistic Regression)
# 'classifier__' prefix targets hyperparameters of the 'classifier' step in the pipeline.
lr_grid_params = {
    'classifier__C': [0.01, 0.1, 1, 10, 100], # C: Inverse of regularization strength. Smaller C means stronger regularization.
    'classifier__solver': ['liblinear', 'lbfgs'] # Solver: Algorithm to use for optimization. 'liblinear' is good for small datasets.
}
print(f"\nLogistic Regression Grid Search Parameter Grid: {lr_grid_params}")

# 3. Perform GridSearchCV for Logistic Regression
# GridSearchCV exhaustively searches all combinations. cv=5 specifies 5-fold cross-validation.
# scoring='accuracy' means it will choose the best parameters based on cross-validation accuracy.
# n_jobs=-1 uses all available CPU cores for faster execution. verbose=1 shows progress.
grid_search_lr = GridSearchCV(lr_pipeline, lr_grid_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
print("Starting GridSearchCV for Logistic Regression with 5-fold cross-validation...")
grid_search_lr.fit(X_train, y_train) # Fit on the training data (internal CV happens here)
print("GridSearchCV for Logistic Regression completed.")

# Report best parameters and cross-validation score from Grid Search
print(f"\nBest parameters for Logistic Regression (Grid Search CV): {grid_search_lr.best_params_}")
print(f"Best cross-validation accuracy (Grid Search CV): {grid_search_lr.best_score_:.4f}")

# Evaluate the best model (retrained on entire X_train with best params) on the unseen X_test
lr_grid_tuned_metrics = evaluate_model(grid_search_lr.best_estimator_, X_test, y_test, "Logistic Regression (Grid Search Tuned)")

# 4. Define parameter distribution for Randomized Search (Logistic Regression)
# loguniform is useful for 'C' as its effect is often multiplicative/logarithmic.
lr_random_params = {
    'classifier__C': loguniform(0.001, 1000), # Random values for C from a log-uniform distribution
    'classifier__solver': ['liblinear', 'lbfgs', 'saga'] # Discrete choices for solver, 'saga' added for broader options
}

print(f"\nLogistic Regression Randomized Search Parameter Distribution: {lr_random_params}")

# 5. Perform RandomizedSearchCV for Logistic Regression
# RandomizedSearchCV samples 'n_iter' combinations randomly. Here, 50 random combinations.
random_search_lr = RandomizedSearchCV(lr_pipeline, lr_random_params, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
print("Starting RandomizedSearchCV for Logistic Regression with 5-fold cross-validation...")
random_search_lr.fit(X_train, y_train) # Fit on the training data
print("RandomizedSearchCV for Logistic Regression completed.")

# Report best parameters and cross-validation score from Randomized Search
print(f"\nBest parameters for Logistic Regression (Randomized Search CV): {random_search_lr.best_params_}")
print(f"Best cross-validation accuracy (Randomized Search CV): {random_search_lr.best_score_:.4f}")

# Evaluate the best model on the unseen X_test
lr_random_tuned_metrics = evaluate_model(random_search_lr.best_estimator_, X_test, y_test, "Logistic Regression (Randomized Search Tuned)")

# 6. Evaluate Logistic Regression with default parameters (for baseline comparison)
# This model uses the default settings of LogisticRegression, without any explicit tuning.
default_lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                      ('classifier', LogisticRegression(random_state=42))])
print("\nTraining Logistic Regression with default parameters (for baseline comparison)...")
default_lr_pipeline.fit(X_train, y_train)
default_lr_metrics = evaluate_model(default_lr_pipeline, X_test, y_test, "Logistic Regression (Default Parameters)")


# --- Decision Tree Classifier Optimization ---
print("\n\n### Decision Tree Classifier Hyperparameter Tuning ###")

# 1. Create a pipeline for Decision Tree Classifier
dt_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                             ('classifier', DecisionTreeClassifier(random_state=42))])
print("Decision Tree pipeline created (Preprocessing + Classifier).")

# 2. Define parameter grid for Grid Search (Decision Tree)
dt_grid_params = {
    'classifier__max_depth': [3, 5, 7, 9, None], # Max depth of the tree. None means unlimited depth (can overfit).
    'classifier__min_samples_leaf': [1, 5, 10, 20], # Minimum number of samples required to be at a leaf node. Higher values prune the tree.
    'classifier__criterion': ['gini', 'entropy'] # Function to measure the quality of a split.
}
print(f"\nDecision Tree Grid Search Parameter Grid: {dt_grid_params}")

# 3. Perform GridSearchCV for Decision Tree
grid_search_dt = GridSearchCV(dt_pipeline, dt_grid_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
print("Starting GridSearchCV for Decision Tree with 5-fold cross-validation...")
grid_search_dt.fit(X_train, y_train)
print("GridSearchCV for Decision Tree completed.")

# Report best parameters and CV score from Grid Search
print(f"\nBest parameters for Decision Tree (Grid Search CV): {grid_search_dt.best_params_}")
print(f"Best cross-validation accuracy (Grid Search CV): {grid_search_dt.best_score_:.4f}")

# Evaluate the best model on the unseen X_test
dt_grid_tuned_metrics = evaluate_model(grid_search_dt.best_estimator_, X_test, y_test, "Decision Tree (Grid Search Tuned)")

# 4. Define parameter distribution for Randomized Search (Decision Tree)
dt_random_params = {
    'classifier__max_depth': randint(3, 20), # Random integer depth between 3 (inclusive) and 20 (exclusive)
    'classifier__min_samples_leaf': randint(1, 20), # Random integer min samples per leaf between 1 (inclusive) and 20 (exclusive)
    'classifier__criterion': ['gini', 'entropy']
}
print(f"\nDecision Tree Randomized Search Parameter Distribution: {dt_random_params}")

# 5. Perform RandomizedSearchCV for Decision Tree
random_search_dt = RandomizedSearchCV(dt_pipeline, dt_random_params, n_iter=50, cv=5, scoring='accuracy', random_state=42, n_jobs=-1, verbose=1)
print("Starting RandomizedSearchCV for Decision Tree with 5-fold cross-validation...")
random_search_dt.fit(X_train, y_train)
print("RandomizedSearchCV for Decision Tree completed.")

# Report best parameters and CV score from Randomized Search
print(f"\nBest parameters for Decision Tree (Randomized Search CV): {random_search_dt.best_params_}")
print(f"Best cross-validation accuracy (Randomized Search CV): {random_search_dt.best_score_:.4f}")

# Evaluate the best model on the unseen X_test
dt_random_tuned_metrics = evaluate_model(random_search_dt.best_estimator_, X_test, y_test, "Decision Tree (Randomized Search Tuned)")

# 6. Evaluate Decision Tree with default parameters (for baseline comparison)
default_dt_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                     ('classifier', DecisionTreeClassifier(random_state=42))])
print("\nTraining Decision Tree with default parameters (for baseline comparison)...")
default_dt_pipeline.fit(X_train, y_train)
default_dt_metrics = evaluate_model(default_dt_pipeline, X_test, y_test, "Decision Tree (Default Parameters)")


# --- 4. Detailed Report on Performance Improvement ---
print("\n" + "="*50)
print("            PERFORMANCE IMPROVEMENT REPORT           ")
print("="*50)

print("\n--- Logistic Regression Performance Comparison ---")
print(f"Metric      | Default     | Grid Tuned  | Random Tuned")
print(f"------------|-------------|-------------|-------------")
print(f"Accuracy    | {default_lr_metrics['accuracy']:.4f}  | {lr_grid_tuned_metrics['accuracy']:.4f}  | {lr_random_tuned_metrics['accuracy']:.4f}")
print(f"Precision   | {default_lr_metrics['precision']:.4f}  | {lr_grid_tuned_metrics['precision']:.4f}  | {lr_random_tuned_metrics['precision']:.4f}")
print(f"Recall      | {default_lr_metrics['recall']:.4f}  | {lr_grid_tuned_metrics['recall']:.4f}  | {lr_random_tuned_metrics['recall']:.4f}")
print(f"F1-Score    | {default_lr_metrics['f1_score']:.4f}  | {lr_grid_tuned_metrics['f1_score']:.4f}  | {lr_random_tuned_metrics['f1_score']:.4f}")
print(f"ROC-AUC     | {default_lr_metrics['roc_auc']:.4f}  | {lr_grid_tuned_metrics['roc_auc']:.4f}  | {lr_random_tuned_metrics['roc_auc']:.4f}")

print("\n--- Decision Tree Performance Comparison ---")
print(f"Metric      | Default     | Grid Tuned  | Random Tuned")
print(f"------------|-------------|-------------|-------------")
print(f"Accuracy    | {default_dt_metrics['accuracy']:.4f}  | {dt_grid_tuned_metrics['accuracy']:.4f}  | {dt_random_tuned_metrics['accuracy']:.4f}")
print(f"Precision   | {default_dt_metrics['precision']:.4f}  | {dt_grid_tuned_metrics['precision']:.4f}  | {dt_random_tuned_metrics['precision']:.4f}")
print(f"Recall      | {default_dt_metrics['recall']:.4f}  | {dt_grid_tuned_metrics['recall']:.4f}  | {dt_random_tuned_metrics['recall']:.4f}")
print(f"F1-Score    | {default_dt_metrics['f1_score']:.4f}  | {dt_grid_tuned_metrics['f1_score']:.4f}  | {dt_random_tuned_metrics['f1_score']:.4f}")
print(f"ROC-AUC     | {default_dt_metrics['roc_auc']:.4f}  | {dt_grid_tuned_metrics['roc_auc']:.4f}  | {dt_random_tuned_metrics['roc_auc']:.4f}")

print("\n--- Best Parameters Found by Optimization ---")
print(f"Logistic Regression (Grid Search): {grid_search_lr.best_params_}")
print(f"Logistic Regression (Randomized Search): {random_search_lr.best_params_}")
print(f"Decision Tree (Grid Search): {grid_search_dt.best_params_}")
print(f"Decision Tree (Randomized Search): {random_search_dt.best_params_}")

print("\n--- Summary of Performance Improvement ---")
print("Tuning hyperparameters generally leads to improved model performance, "
      "as demonstrated by comparing the default models with their Grid Search "
      "and Randomized Search optimized counterparts.")

# Provide specific percentage improvements (calculated on Accuracy)
if lr_grid_tuned_metrics['accuracy'] > default_lr_metrics['accuracy']:
    print(f"\nLogistic Regression Grid Search improved accuracy by: "
          f"{(lr_grid_tuned_metrics['accuracy'] - default_lr_metrics['accuracy'])*100:.2f}%")
else:
    print("\nLogistic Regression Grid Search did not significantly improve accuracy over default.")

if lr_random_tuned_metrics['accuracy'] > default_lr_metrics['accuracy']:
    print(f"Logistic Regression Randomized Search improved accuracy by: "
          f"{(lr_random_tuned_metrics['accuracy'] - default_lr_metrics['accuracy'])*100:.2f}%")
else:
    print("Logistic Regression Randomized Search did not significantly improve accuracy over default.")

if dt_grid_tuned_metrics['accuracy'] > default_dt_metrics['accuracy']:
    print(f"\nDecision Tree Grid Search improved accuracy by: "
          f"{(dt_grid_tuned_metrics['accuracy'] - default_dt_metrics['accuracy'])*100:.2f}%")
else:
    print("\nDecision Tree Grid Search did not significantly improve accuracy over default.")

if dt_random_tuned_metrics['accuracy'] > default_dt_metrics['accuracy']:
    print(f"Decision Tree Randomized Search improved accuracy by: "
          f"{(dt_random_tuned_metrics['accuracy'] - default_dt_metrics['accuracy'])*100:.2f}%")
else:
    print("Decision Tree Randomized Search did not significantly improve accuracy over default.")

print("\nIn many cases, the tuned models show better scores across metrics like "
      "Accuracy, Precision, Recall, F1-Score, and ROC-AUC, indicating their "
      "ability to generalize better to unseen data. Both Grid Search and "
      "Randomized Search are effective for this purpose, with Randomized Search "
      "often being more efficient for larger search spaces, especially when exploring "
      "a wide range of values or many hyperparameters.")

Libraries imported successfully.
Actual Titanic dataset loaded from 'titanic.csv'.
Dataset shape: (891, 12)
First 5 rows of the dataset:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 31012